In [1]:
import torch
import numpy as np
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
import torch
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def load_pcd_from_pth(pth_file, intrinsics, mat):
    # --- Your Logic: Load and Prepare Depth ---
    data = torch.load(pth_file, map_location='cpu')
    width, height = 1920, 1080
    
    flat_array = data.numpy()
    reshaped_tensor = flat_array.reshape((3, height, width))
    depth_2d = reshaped_tensor[0, :, :]
    
    depth_unormalized = depth_2d * 4.0
    depth_clean = np.ascontiguousarray(depth_unormalized.astype(np.float32))
    depth_image = o3d.geometry.Image(depth_clean)
    
    # --- Your Logic: PCD Generation ---
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        depth_image,
        intrinsic=intrinsics,
        extrinsic=mat, # Using the matrix provided in your logic
        depth_scale=1.0,
        depth_trunc=4.0
    )
    
    # --- ADDED: Colorization Logic ---
    points = np.asarray(pcd.points)
    if points.size > 0:
        # We use the Z values (distance) to determine the color
        z_vals = points[:, 2]
        z_min, z_max = z_vals.min(), z_vals.max()
        
        # Normalize Z values to [0, 1] for the colormap
        # Added a small epsilon to avoid division by zero
        z_norm = (z_vals - z_min) / (z_max - z_min + 1e-7)
        
        # Apply Turbo colormap: Sliced to [:, :3] to remove alpha
        # Cast to float64 to ensure Open3D compatibility
        colors = plt.get_cmap("turbo")(z_norm)[:, :3]
        pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))
        
    return pcd

# --- Camera 1 Data ---
pth_file_1 = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D003L.pth"
intrinsics_1 = o3d.camera.PinholeCameraIntrinsic(1920, 1080, 1119.2777099609375, 1119.658447265625, 945.4937133789062, 532.146240234375)
m1 = np.array([
    -0.579134459286479, -0.01693084656633629, 0.8150562094122744, 0.09713516826800955,
    0.06268325847976053, 0.9958997345672709, 0.06522674141178506, 1.0789535394273904,
    -0.8128186065619546, 0.08886543266865657, -0.5756985736505653, 2.598494050623725,
    0.0, 0.0, 0.0, 1.0
]).reshape((4, 4))

# --- Camera 2 Data ---
pth_file_2 = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D005Z.pth"
intrinsics_2 = o3d.camera.PinholeCameraIntrinsic(1920, 1080, 1116.0556640625, 1116.6988525390625, 958.1214599609375, 530.5521850585938)
m2 = np.array([
    -0.07078375138654622, 0.014718319373059337, 0.9973830917027221, -0.21615626613369682,
    0.12154822297481715, 0.9925672655176168, -0.006021039324372706, 0.9671959869175359,
    -0.9900584275846108, 0.12080395067096993, -0.07204662012179419, 2.6001066590136883,
    0.0, 0.0, 0.0, 1.0
]).reshape((4, 4))

# Process both
pcd1 = load_pcd_from_pth(pth_file_1, intrinsics_1, m1)
pcd2 = load_pcd_from_pth(pth_file_2, intrinsics_2, m2)

# --- Visualization ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="Colored Merged Cloud", width=1280, height=720)
vis.add_geometry(pcd1)
vis.add_geometry(pcd2)
vis.add_geometry(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5))

opt = vis.get_render_option()
opt.point_size = 2.0
opt.light_on = False  # Set to False so we see the raw colormap colors
opt.background_color = np.asarray([0.1, 0.1, 0.1])

vis.run()
vis.destroy_window()

In [ ]:

def load_pcd(pth, intrinsic, mat):
    depth = torch.load(pth, map_location='cpu').numpy().reshape((3, 1080, 1920))[0] * 4.0
    
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        o3d.geometry.Image(depth.astype(np.float32)),
        intrinsic=intrinsic,
        extrinsic=mat,
        depth_trunc=4.0
    )
    
    z = np.asarray(pcd.points)[:, 2]
    colors = plt.get_cmap("turbo")((z - z.min()) / (z.max() - z.min() + 1e-7))[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))
    return pcd

m1 = np.array([
    [-0.579134, -0.016931, 0.815056, 0.097135],
    [0.062683, 0.995900, 0.065227, 1.078954],
    [-0.812819, 0.088865, -0.575699, 2.598494],
    [0.0, 0.0, 0.0, 1.0]
])

m2 = np.array([
    [-0.070784, 0.014718, 0.997383, -0.216156],
    [0.121548, 0.992567, -0.006021, 0.967196],
    [-0.990058, 0.120804, -0.072047, 2.600107],
    [0.0, 0.0, 0.0, 1.0]
])

intr1 = o3d.camera.PinholeCameraIntrinsic(1920, 1080, 1119.277, 1119.658, 945.494, 532.146)
intr2 = o3d.camera.PinholeCameraIntrinsic(1920, 1080, 1116.056, 1116.699, 958.121, 530.552)

p1 = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D003L.pth"
p2 = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D005Z.pth"

pcd1 = load_pcd(p1, intr1, m1)
pcd2 = load_pcd(p2, intr2, m2)

vis = o3d.visualization.Visualizer()
vis.create_window()
vis.get_render_option().point_size = 2.0
vis.get_render_option().light_on = False
vis.get_render_option().background_color = [0.1, 0.1, 0.1]
for p in [pcd1, pcd2, o3d.geometry.TriangleMesh.create_coordinate_frame(0.5)]:
    vis.add_geometry(p)
vis.run()

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_auto_zoom(bin_path, w, h):
    # 1. Load data
    depth = np.fromfile(bin_path, dtype=np.float32).reshape((h, w))
    
    # 2. Extract ONLY the subject (ignore the 1.0 background)
    mask = depth < 0.9999
    if not np.any(mask):
        print("No subject found in file (all background).")
        return None

    # 3. HEAL THE DEPTH: 
    # We map the tiny 0.9959-0.9999 range into 1.0-2.0 meters 
    # so the 3D features (nose, shoulders) become visible.
    d_min, d_max = depth[mask].min(), depth[mask].max()
    print(f"Mapping subject range [{d_min:.4f} -> {d_max:.4f}] to [1.0m -> 2.0m]")
    
    # Normalized depth for subject only
    depth_rescaled = np.ones_like(depth) * 5.0 # Set background far away
    depth_rescaled[mask] = (depth[mask] - d_min) / (d_max - d_min + 1e-7) + 1.0

    # 4. JSON Intrinsics
    intr = o3d.camera.PinholeCameraIntrinsic(
        w, h, 4564.9038, 4565.1894, 2331.1484, 2639.1892
    )

    # 5. Create PCD (Camera Space)
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        o3d.geometry.Image(depth_rescaled.astype(np.float32)),
        intrinsic=intr,
        depth_scale=1.0,
        depth_trunc=2.1 # Cut off the 5.0 background
    )

    # 6. Center the subject at the origin [0,0,0] for easy viewing
    pcd.translate(-pcd.get_center())

    # 7. Colorize & Downsample
    pcd = pcd.voxel_down_sample(voxel_size=0.002)
    z_vals = np.asarray(pcd.points)[:, 2]
    z_norm = (z_vals - z_vals.min()) / (z_vals.max() - z_vals.min() + 1e-7)
    pcd.colors = o3d.utility.Vector3dVector(plt.get_cmap("turbo")(z_norm)[:, :3])

    return pcd

bin_file = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/export_depth_eimer_non_partial/frame_00000/depth/depth_0.bin"
pcd = visualize_auto_zoom(bin_file, 4608, 5328)

if pcd:
    # Use the simple draw_geometries to allow manual rotation
    # If it still looks flat, right-click and rotate the view 90 degrees!
    o3d.visualization.draw_geometries([pcd], 
                                      window_name="Subject Only - Rotate to see 3D",
                                      width=1280, height=720,
                                      left=50, top=50,
                                      mesh_show_back_face=True)

In [1]:
import torch
import numpy as np
import open3d as o3d
import cv2
import matplotlib.pyplot as plt

def load_sensor_pth(pth_path):
    data = torch.load(pth_path, map_location='cpu')
    return data.numpy().reshape((3, 1080, 1920))[0] * 4.0

def visualize_limited_carve(vh_bin_path, sensor_pth_path):
    w, h = 4608, 5328
    vh_raw = np.fromfile(vh_bin_path, dtype=np.float32).reshape((h, w))
    
    # 1. Subject Mask
    mask = vh_raw < 0.9999
    v, u = np.where(mask)
    
    # 2. Rescale Hull to real meters (2.0m to 2.8m range)
    z_vh = 2.0 + (vh_raw[mask] - 0.9959) / (1.0 - 0.9959) * 0.8
    
    # 3. Load Sensor
    sensor_small = load_sensor_pth(sensor_pth_path)
    sensor_resized = cv2.resize(sensor_small, (w, h), interpolation=cv2.INTER_NEAREST)
    z_sensor = sensor_resized[mask]
    
    # 4. SURGICAL CARVING WITH A CEILING
    # epsilon = how much deeper the sensor must be to start carving (3cm)
    # max_carve = the maximum depth of the box (20cm). Anything deeper is ignored.
    epsilon = 0.03
    max_carve = 0.20 
    
    # Condition: 
    # 1. Sensor must be valid (> 0.5m)
    # 2. Sensor must be deeper than Hull + epsilon
    # 3. Sensor must NOT be deeper than Hull + max_carve (prevents carving the body away)
    carve_mask = (z_sensor > 0.5) & \
                 (z_sensor > z_vh + epsilon) & \
                 (z_sensor < z_vh + max_carve)
    
    z_final = np.copy(z_vh)
    z_final[carve_mask] = z_sensor[carve_mask]

    # 5. Manual 3D Projection
    fx, fy, cx, cy = 4564.9, 4565.1, 2331.1, 2639.1
    x_3d = (u - cx) * z_final / fx
    y_3d = (v - cy) * z_final / fy
    z_3d = z_final
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.stack([x_3d, y_3d, z_3d], axis=1))
    
    # 6. Heatmap + Carve Highlight
    # Base heatmap: Green (Front) to Blue (Back)
    z_norm = (z_3d - z_3d.min()) / (z_3d.max() - z_3d.min() + 1e-7)
    colors = plt.get_cmap("turbo")(z_norm)[:, :3]
    
    # Make the CARVED points (the box hole) White/Gold
    colors[carve_mask] = [1.0, 0.9, 0.4] 
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    pcd.translate(-pcd.get_center())
    return pcd

# --- PATHS ---
vh_path = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/export_depth_eimer_non_partial/frame_00000/depth/depth_0.bin"
sensor_path = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D003L.pth"

pcd = visualize_limited_carve(vh_path, sensor_path)

# --- VIEW CONTROLS ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="Turbo=Body, Gold=Concave Box", width=1280, height=720)
vis.add_geometry(pcd)

# This frames the object perfectly in the center
vis.reset_view_point(True)

opt = vis.get_render_option()
opt.background_color = [0, 0, 0]
opt.point_size = 2.0

# Start with a slight tilt so you can see the depth
ctr = vis.get_view_control()
ctr.set_front([-0.3, 0, -0.9]) 

vis.run()
vis.destroy_window()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
import torch
import numpy as np
import open3d as o3d
import cv2
import matplotlib.pyplot as plt

def load_sensor_pth(pth_path):
    data = torch.load(pth_path, map_location='cpu')
    return data.numpy().reshape((3, 1080, 1920))[0] * 4.0

def visualize_dual_carve(vh_bin_path, sensor1_path, sensor2_path):
    w, h = 4608, 5328
    vh_raw = np.fromfile(vh_bin_path, dtype=np.float32).reshape((h, w))
    
    # 1. Subject Mask
    mask = vh_raw < 0.9999
    v, u = np.where(mask)
    
    # 2. Rescale Hull (2.0m to 2.8m)
    z_vh = 2.0 + (vh_raw[mask] - 0.9959) / (1.0 - 0.9959) * 0.8
    
    # 3. Load and Resize BOTH Sensors
    s1_small = load_sensor_pth(sensor1_path)
    s2_small = load_sensor_pth(sensor2_path)
    
    s1_resized = cv2.resize(s1_small, (w, h), interpolation=cv2.INTER_NEAREST)[mask]
    s2_resized = cv2.resize(s2_small, (w, h), interpolation=cv2.INTER_NEAREST)[mask]
    
    # 4. MULTI-SENSOR CARVING LOGIC
    epsilon = 0.03
    max_carve = 0.25 # Max depth of the box hole
    
    # Camera 1's opinion
    carve1 = (s1_resized > 0.5) & (s1_resized > z_vh + epsilon) & (s1_resized < z_vh + max_carve)
    # Camera 2's opinion
    carve2 = (s2_resized > 0.5) & (s2_resized > z_vh + epsilon) & (s2_resized < z_vh + max_carve)
    
    # COMBINE: Carve if either camera sees the hole
    combined_carve_mask = carve1 | carve2
    
    z_final = np.copy(z_vh)
    # For the final depth, we take the DEEPEST valid sensor value to get the full concavity
    z_final[combined_carve_mask] = np.maximum(s1_resized[combined_carve_mask], s2_resized[combined_carve_mask])

    # 5. 3D Projection
    fx, fy, cx, cy = 4564.9, 4565.1, 2331.1, 2639.1
    x_3d = (u - cx) * z_final / fx
    y_3d = (v - cy) * z_final / fy
    z_3d = z_final
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.stack([x_3d, y_3d, z_3d], axis=1))
    
    # 6. Color Mapping
    z_norm = (z_3d - z_3d.min()) / (z_3d.max() - z_3d.min() + 1e-7)
    colors = plt.get_cmap("turbo")(z_norm)[:, :3]
    
    # GOLD highlight for the combined carved area
    colors[combined_carve_mask] = [1.0, 0.9, 0.2] 
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    pcd.translate(-pcd.get_center())
    return pcd

# --- PATHS (Change sensor2_path to your second camera file) ---
vh_path = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/export_depth_eimer_non_partial/frame_00000/depth/depth_0.bin"
sensor1_path = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D003L.pth"
sensor2_path = "/home/timnogga/bachelorthesis/biwi_kinect_head_pose/RIFTCast/data/2026_03_20_orbbec_002_standard/frame_00000/rgb/D005Z.pth" 

pcd = visualize_dual_carve(vh_path, sensor1_path, sensor2_path)

# --- VISUALIZER ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="Dual-Sensor Carving", width=1280, height=720)
vis.add_geometry(pcd)
vis.reset_view_point(True)
vis.get_render_option().point_size = 2.5
vis.get_render_option().background_color = [0, 0, 0]
vis.run()
vis.destroy_window()